<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Lupa que se Desliza: Cómo «Ve» una Computadora 🔍
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/09%20-%20Casos%20de%20Estudio%20y%20Aplicaciones/Para%20Dummies/01_Vision_Computador_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 01 del Módulo 09 (Visión por computador)](../01_Vision_por_Computador.ipynb). Para una computadora, una foto **no es un paisaje: es una tabla de números** (el brillo de cada puntito). Para «ver» algo, pasa una **lupa pequeña** por toda la tabla buscando **patrones** (bordes, esquinas, manchas), como cuando buscas una palabra recorriendo un texto con el dedo.

Al terminar podrás explicar:
1. Qué es una **imagen** para una computadora y qué hace la **plantilla deslizante** (la convolución).
2. Cómo se mide **«qué tanto se pisan dos cajas»** (IoU) y por qué hay que **descartar las cajas repetidas** (NMS) cuando se detectan objetos.
3. Por qué un modelo que acierta mucho **puede fallar** si el objeto se corre un poquito, y cómo se arregla.

> 🔗 Si luego quieres ver las versiones **desde cero y verificadas** (con PyTorch, detección de objetos y mAP): [cuaderno estándar](../01_Vision_por_Computador.ipynb). Antes de esto: [El triage de urgencias](00_Casos_Estudio_Dummies.ipynb). Después: [Contar palabras como cartas](02_NLP_Dummies.ipynb).

---
## 1. La imagen es una tabla y la lupa es una plantilla 🔍

Imagina una foto diminuta de **6×6 puntitos**, con números del 0 (oscuro) al 9 (claro): la mitad izquierda es oscura y la derecha, clara; entre ambas hay un **borde vertical**. Para encontrarlo usamos una **plantilla** de 3×3 casillas con números: **−1** a la izquierda (resta lo oscuro) y **+1** a la derecha (suma lo claro). La **deslizamos** por toda la imagen: en cada lugar **multiplicamos casilla por casilla y sumamos**. Si todo es parejo, la suma es **0**; si hay un salto de oscuro a claro, la suma es **grande**. Eso es una **convolución**, la operación base de las redes que reconocen imágenes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# --- Una imagen diminuta de 6x6 "pixeles": 0 = oscuro, 9 = claro. Mitad izquierda oscura, mitad derecha clara ---
imagen = np.array([[0, 0, 0, 9, 9, 9]] * 6, dtype=float)
# La "lupa": una plantilla 3x3 que compara el lado derecho con el izquierdo (detecta bordes verticales)
plantilla = np.array([[-1, 0, 1],
                      [-1, 0, 1],
                      [-1, 0, 1]], dtype=float)

# Deslizamos la plantilla por toda la imagen: en cada posicion multiplicamos casilla por casilla y sumamos
resultado = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        ventana = imagen[i:i + 3, j:j + 3]
        resultado[i, j] = np.sum(ventana * plantilla)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, M, titulo in [(axes[0], imagen, "La imagen (números = brillo)"), (axes[1], plantilla, "La plantilla (la «lupa»)"), (axes[2], resultado, "Dónde encontró un borde vertical")]:
    ax.imshow(M, cmap="gray" if M is imagen else "RdBu_r"); ax.set_title(titulo, fontweight="bold", fontsize=10)
    for (i, j), v in np.ndenumerate(M):
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", color="red" if M is imagen else "black", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("Resultado de deslizar la lupa:\n", resultado)
print("→ Solo en las columnas del medio (donde pasa de oscuro a claro) el número es grande. En el resto, 0: ahí no hay borde.")
assert resultado[:, 1].min() == 27 and resultado[:, 0].max() == 0 and resultado[:, 3].max() == 0

---
**Lección:** la plantilla se «enciende» (27) **solo donde hay un borde vertical** y vale **0** donde la imagen es pareja. Una red neuronal para imágenes usa **muchas plantillas** de este tipo, pero en lugar de que las diseñe una persona, **las aprende sola** mirando miles de ejemplos: unas aprenden bordes, otras esquinas, otras texturas, y las siguientes combinan esas piezas en **ojos, ruedas o letras**.

---

## 2. Dónde está el objeto: cajas, IoU y quedarse con la mejor 📦

Reconocer **qué** hay en una foto es una cosa; encontrar **dónde** está, otra: se dibuja una **caja** alrededor de cada objeto. Para saber si una caja es buena se compara con la **correcta** usando el **IoU** (intersección sobre unión): imagina **dos manteles** sobre una mesa; el IoU es *«el pedazo donde se pisan, dividido por todo lo que cubren entre los dos»*. Vale **1** si coinciden y **0** si no se tocan.

Además, un detector suele **dibujar varias cajas casi iguales alrededor del mismo gato**. Para quedarse con una se hace como al elegir **un solo representante por grupo**: se toma la caja **más segura**, se descartan las que se le **pisan demasiado** y se repite con las que quedan. Eso se llama **supresión de no máximos (NMS)**.

In [ ]:
# Dos cajas (dos manteles sobre una mesa). ¿Que tanto se pisan? IoU = area donde se pisan / area total que cubren entre las dos
def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    interseccion = max(0, x2 - x1) * max(0, y2 - y1)
    area_a, area_b = (a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1])
    return interseccion / (area_a + area_b - interseccion)


real = (2, 2, 8, 8)                                              # la caja correcta alrededor del gato
intentos = {"casi perfecta": (2.5, 2.5, 8.5, 8.5), "corrida a medias": (5, 5, 11, 11), "en otro lugar": (9, 9, 14, 14)}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (nombre, caja) in zip(axes, intentos.items()):
    ax.add_patch(mpatches.Rectangle(real[:2], 6, 6, fill=False, ec="#16a34a", lw=3, label="caja correcta"))
    ax.add_patch(mpatches.Rectangle(caja[:2], caja[2] - caja[0], caja[3] - caja[1], fill=False, ec="#dc2626", lw=2, ls="--", label="intento"))
    ax.set_xlim(0, 15); ax.set_ylim(15, 0); ax.set_aspect("equal"); ax.set_title(f"{nombre}: IoU = {iou(real, caja):.2f}", fontweight="bold", fontsize=10)
axes[0].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()
print({k: round(iou(real, v), 2) for k, v in intentos.items()})
print("Regla habitual: si el IoU es de 0.5 o más, contamos el intento como un ACIERTO.")
assert iou(real, real) == 1 and iou(real, intentos["en otro lugar"]) == 0 and 0.7 < iou(real, intentos["casi perfecta"]) < 0.9

# --- Muchas cajas para el MISMO gato: nos quedamos con la mas segura y descartamos las que se le pisan demasiado ---
cajas = [(2, 2, 8, 8), (2.5, 2, 8.5, 8.5), (1.5, 2.5, 7.5, 8), (3, 3, 9, 9), (10, 10, 14, 14)]      # las 4 primeras son del mismo gato; la 5.ª es otro objeto
confianza = [0.95, 0.90, 0.80, 0.60, 0.85]
orden = np.argsort(confianza)[::-1]
quedan = []
for i in orden:
    if all(iou(cajas[i], cajas[j]) <= 0.5 for j in quedan):                                          # solo la dejamos si no se pisa mucho con una ya aceptada
        quedan.append(i)
print(f"De {len(cajas)} cajas nos quedamos con {len(quedan)}: las números {sorted(int(i) for i in quedan)} (la 0 es la más segura del gato; la 4 es otro objeto).")
assert sorted(quedan) == [0, 4]

---
**Lección:** un intento «casi perfecto» tiene IoU ≈ 0.72 (acierto), uno «corrido a medias» solo ≈ 0.14 y uno «en otro lugar», 0. Y de **5 cajas** (4 del mismo gato y 1 de otro objeto) nos quedamos con **2**: la mejor del gato y la del otro objeto. Sin NMS, el detector contaría **cuatro gatos** donde hay uno.

---

## 3. El modelo que memorizó el lugar, no la forma 🔢

Probemos un modelo sencillo que reconoce **dígitos escritos a mano** (imágenes de 8×8). Acierta casi todo. Pero, ¿qué pasa si **corremos cada dígito un solo puntito** hacia la derecha? Para nosotros sigue siendo el mismo número; para el modelo, **cada casilla es un dato aparte**, y si la tinta cae en otras casillas... se confunde. Después probaremos un remedio: **mostrarle copias corridas** mientras estudia (se llama *aumento de datos*), como enseñarle una letra con la hoja en distintas posiciones.

In [ ]:
# Un modelo sencillo que reconoce digitos escritos a mano (imagenes de 8x8 pixeles)
d = load_digits()
X = (d.data / 16.0).reshape(-1, 8, 8); y = d.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

def mover(imgs, dx=1):
    """Corre cada imagen un pixel a la derecha (lo que sale se pierde; lo que entra es oscuro)."""
    return np.pad(imgs, ((0, 0), (0, 0), (dx, 0)))[:, :, :8]

modelo = LogisticRegression(max_iter=2000).fit(X_tr.reshape(len(X_tr), -1), y_tr)
a_normal = modelo.score(X_te.reshape(len(X_te), -1), y_te)
a_movido = modelo.score(mover(X_te).reshape(len(X_te), -1), y_te)
print(f"Dígitos normales:               {a_normal:.0%} de aciertos")
print(f"Los mismos dígitos corridos 1 píxel: {a_movido:.0%} de aciertos   ← ¡el modelo aprendió DÓNDE está la tinta, no la forma!")

# Remedio: "aumentar" los datos mostrandole tambien copias corridas (como ensenar con la hoja en distintas posiciones)
X_aum = np.concatenate([X_tr, mover(X_tr), np.pad(X_tr, ((0, 0), (0, 0), (0, 1)))[:, :, 1:]]); y_aum = np.tile(y_tr, 3)
modelo2 = LogisticRegression(max_iter=2000).fit(X_aum.reshape(len(X_aum), -1), y_aum)
print(f"Con copias corridas al estudiar:   normales {modelo2.score(X_te.reshape(len(X_te), -1), y_te):.0%} | corridos {modelo2.score(mover(X_te).reshape(len(X_te), -1), y_te):.0%}")
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for k in range(2):
    axes[2 * k].imshow(X_te[k], cmap="gray_r"); axes[2 * k].set_title("original", fontsize=9)
    axes[2 * k + 1].imshow(mover(X_te[k:k + 1])[0], cmap="gray_r"); axes[2 * k + 1].set_title("corrido 1 píxel", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()
assert a_normal > 0.9 and a_movido < a_normal - 0.2

---
**Lección:** con los dígitos de siempre acierta ≈ 97 %, pero **corridos un solo píxel** baja a ≈ 39 %: **aprendió dónde está la tinta, no cuál es la forma**. Mostrarle copias corridas lo **arregla en buena parte** (≈ 90 % con los corridos), pero **le cuesta algo** en lo normal (≈ 92 %): un modelo tan simple no tiene espacio para ser flexible en todo. Las **redes convolucionales** (las de las plantillas deslizantes) reparten mejor ese trabajo y, con aumento, consiguen ser robustas **sin** perder aciertos: eso se ve en el cuaderno estándar. Y una advertencia honesta: el aumento ayuda **cuando las variaciones que enseñas son las que habrá en la vida real**; no es magia.

---

## Resumen en una frase 🎯

> **Una computadora «ve» pasando plantillas por una tabla de números; para ubicar objetos mide cuánto se pisan las cajas (IoU) y se queda con la más segura de cada grupo (NMS), y para no confundirse cuando algo se corre, hay que enseñarle ejemplos corridos.**

### Lo que aprendiste hoy:
- ✅ **Una imagen es una tabla de números**; la **convolución** desliza una plantilla y se «enciende» donde está el patrón (aquí, un borde vertical).
- ✅ **IoU** = pedazo en que se pisan dos cajas ÷ lo que cubren entre las dos (1 = iguales, 0 = no se tocan); un IoU ≥ 0.5 suele contar como acierto.
- ✅ **NMS**: de varias cajas sobre el mismo objeto, se queda la más segura y se descartan las que se le pisan.
- ✅ Un modelo puede acertar 97 % y **caer a 39 %** si el objeto se corre un píxel; el **aumento de datos** ayuda, pero **no es gratis**.

> 🎓 **Siguiente paso:** [el cuaderno estándar](../01_Vision_por_Computador.ipynb) lo hace **desde cero y verificado contra PyTorch**, con una CNN, un detector de juguete y la métrica mAP; o sigue con [contar palabras como cartas de una baraja](02_NLP_Dummies.ipynb).

---
##### 🛠️ Práctica 1: ¿Cuánto se puede correr una caja y seguir contando como acierto?

Tenemos la caja correcta `real = (2, 2, 8, 8)` y la función `iou(a, b)` que ya definimos (los «dos manteles»). Vamos a **correr** la caja del detector hacia la derecha, de a poquito, y ver cuándo deja de ser un acierto:

1. Haz una lista `corrimientos = [0, 1, 2, 3]` (cuántas casillas se corre el intento hacia la derecha).
2. Para cada corrimiento crea la caja del intento `(2 + dx, 2, 8 + dx, 8)` y calcula su IoU con `real`; guarda todo en un diccionario `ious` (clave: el corrimiento; valor: el IoU) e imprímelo.
3. Con la regla habitual (IoU de 0.5 o más es acierto), haz una lista `aciertos` con los corrimientos que **sí** cuentan como acierto.
4. Comprueba que con corrimiento 0 el IoU es exactamente 1 y que los aciertos son los corrimientos 0, 1 y 2.

In [ ]:
# Escribe tu código aquí

# corrimientos = [0, 1, 2, 3]
# ious = {}
# for dx in corrimientos:
#     intento = (2 + dx, 2, 8 + dx, 8)
#     ious[dx] = iou(real, ...)
# print(ious)
#
# aciertos = [dx for dx in corrimientos if ious[dx] >= ...]


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
corrimientos = [0, 1, 2, 3]
ious = {}
for dx in corrimientos:
    intento = (2 + dx, 2, 8 + dx, 8)              # la misma caja, corrida dx casillas a la derecha
    ious[dx] = iou(real, intento)
print({dx: round(v, 2) for dx, v in ious.items()})

aciertos = [dx for dx in corrimientos if ious[dx] >= 0.5]
print("Corrimientos que todavía cuentan como acierto:", aciertos)

assert ious[0] == 1
assert aciertos == [0, 1, 2]
print("Con 2 casillas de corrimiento el IoU llega justo a 0.5 (el límite); con 3 ya se pisan muy poco y deja de ser acierto.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué hace la plantilla (la «lupa») cuando la deslizas por una imagen? ¿Por qué da 0 en las zonas parejas y un número grande en un borde?
2. Dos cajas se pisan en un tercio de lo que cubren entre las dos. ¿Cuánto vale el IoU aproximadamente? ¿Contarías el intento como acierto con el criterio de 0.5? ¿Y qué pasaría con el recuento de objetos si no descartáramos las cajas repetidas?
3. El modelo de dígitos pasó de ≈ 97 % a ≈ 39 % al correr la imagen un píxel. ¿Qué te dice eso sobre lo que había aprendido? Propón **otra variación** (aparte de correr) que podría confundir a un modelo de fotos de la calle y cómo la «enseñarías».

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>